# Neurora: Satellite Hotspot Forecast Evaluation
### Chronological Benchmark Evaluation: Baseline vs. Poisson Regression

This reproducible notebook prepares historical satellite fire-detection counts (NASA FIRMS multi-sensor archive), generates chronological time-series features, trains a **Baseline Lag-1 Persistence Model** against an **Alternative Poisson Regression Model**, and computes held-out test evaluation metrics (**MAE**, **RMSE**).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Set style
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Real Historical Satellite Detection Dataset
Load data exported from the Neurora pipeline (containing daily observation counts, data coverage status, and meteorological indicators).

In [ ]:
dataset_path = '../data/historical_fire_counts.csv'
if not os.path.exists(dataset_path):
    # Fallback to local path if executed from root
    dataset_path = 'data/historical_fire_counts.csv'

df_raw = pd.read_csv(dataset_path)
print(f"Loaded dataset with {len(df_raw)} records across {df_raw['geo_id'].nunique()} regions.")
df_raw.head()

## 2. Data Quality & Coverage Integrity
**Critical Methodological Rule:** We explicitly distinguish confirmed zero observations from missing/delayed passes. Missing observations are omitted from baseline computations rather than substituted with zero.

In [ ]:
coverage_breakdown = df_raw['data_coverage_status'].value_counts(normalize=True) * 100
print("Data Coverage Status Breakdown (%):")
print(coverage_breakdown.round(2))

# Filter for Global time series with valid counts
df_global = df_raw[(df_raw['geo_id'] == 'GLOBAL') & (df_raw['daily_detection_count'].notna())].copy()
df_global['date'] = pd.to_datetime(df_global['date'])
df_global.sort_values(by='date', inplace=True)
df_global.reset_index(drop=True, inplace=True)
print(f"\nValid chronological dates for GLOBAL: {len(df_global)} days")

## 3. Chronological Feature Engineering
Generate temporal lag features, rolling statistics, calendar sine/cosine cyclical features, and surface weather variables.

In [ ]:
# Lags
df_global['lag_1'] = df_global['daily_detection_count'].shift(1)
df_global['lag_2'] = df_global['daily_detection_count'].shift(2)
df_global['rolling_mean_7'] = df_global['daily_detection_count'].shift(1).rolling(window=7, min_periods=3).mean()
df_global['rolling_std_7'] = df_global['daily_detection_count'].shift(1).rolling(window=7, min_periods=3).std()

# Cyclical Day of Year
doy = df_global['date'].dt.dayofyear
df_global['sin_doy'] = np.sin(2 * np.pi * doy / 365.25)
df_global['cos_doy'] = np.cos(2 * np.pi * doy / 365.25)

# Weather proxies
df_global['temp_feat'] = df_global['mean_temp_c'].fillna(25.0)
df_global['humidity_feat'] = df_global['mean_humidity_pct'].fillna(50.0)
df_global['wind_feat'] = df_global['mean_wind_kmh'].fillna(12.0)
df_global['precip_feat'] = df_global['total_precip_mm'].fillna(0.0)

# Drop uninitialized rows
df_features = df_global.dropna(subset=['lag_1', 'lag_2', 'rolling_mean_7']).copy()
df_features.reset_index(drop=True, inplace=True)
print(f"Dataset prepared with {len(df_features)} feature-complete daily observations.")

## 4. Chronological Train & Held-Out Test Split
We strictly avoid random shuffling. The evaluation is conducted on a chronological held-out test split (last 20% of the timeline).

In [ ]:
test_ratio = 0.20
split_idx = int(len(df_features) * (1.0 - test_ratio))

train_df = df_features.iloc[:split_idx].copy()
test_df = df_features.iloc[split_idx:].copy()

feature_cols = [
    'lag_1', 'lag_2', 'rolling_mean_7',
    'sin_doy', 'cos_doy',
    'temp_feat', 'humidity_feat', 'wind_feat', 'precip_feat'
]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
y_train = train_df['daily_detection_count'].values

X_test = scaler.transform(test_df[feature_cols])
y_test = test_df['daily_detection_count'].values

print(f"Training set: {train_df['date'].iloc[0].strftime('%Y-%m-%d')} to {train_df['date'].iloc[-1].strftime('%Y-%m-%d')} ({len(train_df)} days)")
print(f"Held-out test set: {test_df['date'].iloc[0].strftime('%Y-%m-%d')} to {test_df['date'].iloc[-1].strftime('%Y-%m-%d')} ({len(test_df)} days)")

## 5. Model Training & Held-Out Evaluation
Compare:
1. **Baseline Persistence Model:** $y(t) = y(t-1)$
2. **Alternative Poisson Regressor:** Log-linear count distribution with L2 regularization.

In [ ]:
# 1. Baseline Model
y_pred_baseline = test_df['lag_1'].values

# 2. Alternative Poisson Model
alt_model = PoissonRegressor(alpha=1e-2, max_iter=500, solver='lbfgs')
alt_model.fit(X_train, y_train)
y_pred_alt = alt_model.predict(X_test)

# Compute Evaluation Metrics
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = root_mean_squared_error(y_test, y_pred_baseline)

alt_mae = mean_absolute_error(y_test, y_pred_alt)
alt_rmse = root_mean_squared_error(y_test, y_pred_alt)

mae_delta_pct = ((baseline_mae - alt_mae) / baseline_mae) * 100.0

print("=" * 60)
print("CHRONOLOGICAL HELD-OUT TEST EVALUATION RESULTS")
print("=" * 60)
print(f"Baseline Persistence:       MAE = {baseline_mae:,.2f} | RMSE = {baseline_rmse:,.2f}")
print(f"Alternative Poisson Model:  MAE = {alt_mae:,.2f} | RMSE = {alt_rmse:,.2f}")
print(f"MAE Improvement:            {mae_delta_pct:+.2f}%")
print("=" * 60)

## 6. Plotting Actuals vs Model Forecasts

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test_df['date'], y_test, label='Observed Actuals', color='#F8FAFC', lw=2.5, marker='o', markersize=4)
plt.plot(test_df['date'], y_pred_baseline, label='Baseline Lag-1 Persistence', color='#94A3B8', linestyle='--', lw=1.5)
plt.plot(test_df['date'], y_pred_alt, label='Alternative Poisson Model', color='#10B981', lw=2)
plt.title('Chronological Held-Out Test Evaluation: Observed vs. Forecasted Daily Detections', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Daily Satellite Detections')
plt.grid(True, alpha=0.15)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Limitations & Scientific Provenance
- **Detections vs Ground Fires:** These forecasts model satellite pixel thermal anomalies (VIIRS 375m and MODIS 1km) under specified confidence thresholds, not confirmed ground fire perimeters.
- **Orbital Geometry & Weather Obscuration:** Heavy clouds, rain, and thick smoke can impede satellite radiometers, causing temporary dips in observable detections that should not be confused with true fire extinguishment.